In [ ]:
!pip install transformers datasets accelerate scikit-learn sentencepiece protobuf -q

In [ ]:
from google.colab import drive
import os, zipfile

drive.mount('/content/drive')

zip_path = "/content/drive/MyDrive/deberta-v3-small.zip"
cache_dir = os.path.expanduser("~/.cache/huggingface/hub/")
os.makedirs(cache_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    for info in z.infolist():
        fixed_name = info.filename.replace("\\", "/")
        out_path = os.path.join(cache_dir, fixed_name)
        if info.is_dir() or fixed_name.endswith("/"):
            os.makedirs(out_path, exist_ok=True)
        elif info.file_size == 0:
            os.makedirs(os.path.dirname(out_path), exist_ok=True)
        else:
            os.makedirs(os.path.dirname(out_path), exist_ok=True)
            with z.open(info) as src, open(out_path, 'wb') as dst:
                dst.write(src.read())

model_cache = os.path.join(cache_dir, "models--microsoft--deberta-v3-small")
print(f"Model extracted to: {model_cache}")
print(f"Contents: {os.listdir(model_cache)}")

Mounted at /content/drive
Model extracted to: /root/.cache/huggingface/hub/models--microsoft--deberta-v3-small
Contents: ['refs', 'blobs', 'snapshots']


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import json
import os
import gc
import re
import nltk
import random
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine_sim

os.environ["HF_HUB_OFFLINE"] = "1"

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split

nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Using device: cuda
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [ ]:
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [w for w in text.split() if w not in stop_words]
    return ' '.join(tokens)

df_raw = pd.read_csv('ExioNAICS.csv')

naics_corpus = df_raw[['NAICS Code', 'NAICS Title', 'Description']].drop_duplicates(subset='NAICS Code').copy()
naics_corpus['NAICS Code'] = naics_corpus['NAICS Code'].astype(str)
naics_corpus['clean_text'] = (naics_corpus['NAICS Title'] + ' ' + naics_corpus['Description'].fillna('')).apply(preprocess_text)
naics_corpus = naics_corpus.reset_index(drop=True)

corpus_texts = naics_corpus['clean_text'].tolist()
code_to_idx = {code: i for i, code in enumerate(naics_corpus['NAICS Code'])}

df = pd.read_csv('ExioNAICS_preprocessed.csv')
df['NAICS Code'] = df['NAICS Code'].astype(str)
df['naics_idx'] = df['NAICS Code'].map(code_to_idx)

missing = df['naics_idx'].isna().sum()
if missing > 0:
    df = df.dropna(subset=['naics_idx']).reset_index(drop=True)
df['naics_idx'] = df['naics_idx'].astype(int)

print(f"Corpus: {len(corpus_texts)} NAICS codes")
print(f"Dataset: {len(df)} samples")

print("\nBuilding TF-IDF index for candidate retrieval...")
tfidf = TfidfVectorizer(max_features=10000, sublinear_tf=True)
all_clean = corpus_texts + df['clean_description'].tolist()
tfidf_matrix = tfidf.fit_transform(all_clean)
corpus_tfidf = tfidf_matrix[:len(corpus_texts)]
query_tfidf = tfidf_matrix[len(corpus_texts):]

print("Computing similarity matrix...")
sim_matrix = sklearn_cosine_sim(query_tfidf, corpus_tfidf)
tfidf_top100 = np.argsort(-sim_matrix, axis=1)[:, :100]

tfidf_r50 = np.mean([df['naics_idx'].iloc[i] in tfidf_top100[i, :50] for i in range(len(df))])
tfidf_r100 = np.mean([df['naics_idx'].iloc[i] in tfidf_top100[i] for i in range(len(df))])
print(f"\nTF-IDF Recall@50:  {tfidf_r50:.4f}")
print(f"TF-IDF Recall@100: {tfidf_r100:.4f}")

del sim_matrix
gc.collect()

Corpus: 1115 NAICS codes
Dataset: 20535 samples

Building TF-IDF index for candidate retrieval...
Computing similarity matrix...

TF-IDF Recall@50:  0.5639
TF-IDF Recall@100: 0.6360


30

In [ ]:
MODEL_NAME = "microsoft/deberta-v3-small"
MAX_LENGTH = 192
BATCH_SIZE = 8
NUM_NEGATIVES = 7
NUM_EPOCHS = 15
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
SEED = 42
VAL_RATIO = 0.1
PATIENCE = 5
EVAL_TOP_K = 50

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print(f"=== DeBERTa Cross-Encoder Reranking ===")
print(f"  Model:       {MODEL_NAME}")
print(f"  Max length:  {MAX_LENGTH} (combined query + doc)")
print(f"  Batch size:  {BATCH_SIZE} queries x {NUM_NEGATIVES+1} candidates = {BATCH_SIZE*(NUM_NEGATIVES+1)} sequences/step")
print(f"  Negatives:   {NUM_NEGATIVES} TF-IDF hard negatives per positive")
print(f"  Epochs:      {NUM_EPOCHS}")
print(f"  LR:          {LEARNING_RATE}")
print(f"  Patience:    {PATIENCE}")
print(f"  Eval top-k:  {EVAL_TOP_K} TF-IDF candidates")

=== DeBERTa Cross-Encoder Reranking ===
  Model:       microsoft/deberta-v3-small
  Max length:  192 (combined query + doc)
  Batch size:  8 queries x 8 candidates = 64 sequences/step
  Negatives:   7 TF-IDF hard negatives per positive
  Epochs:      15
  LR:          2e-05
  Patience:    5
  Eval top-k:  50 TF-IDF candidates


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=1, torch_dtype=torch.float32
).to(device)

print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

test_enc = tokenizer("company description", "naics industry description", return_tensors="pt", max_length=MAX_LENGTH, truncation=True, padding=True)
test_enc = {k: v.to(device) for k, v in test_enc.items() if k in ['input_ids', 'attention_mask']}
with torch.no_grad():
    test_out = model(**test_enc)
print(f"Output shape: {test_out.logits.shape}")
print(f"NaN: {torch.isnan(test_out.logits).any().item()}")

The tokenizer you are loading from 'microsoft/deberta-v3-small' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight     

Total params: 141,895,681
Output shape: torch.Size([1, 1])
NaN: False


In [ ]:
class CrossEncoderDataset(Dataset):
    def __init__(self, descriptions, naics_indices, corpus_texts, tfidf_candidates, tokenizer, max_length, num_negatives=7):
        self.descriptions = descriptions
        self.naics_indices = naics_indices
        self.corpus_texts = corpus_texts
        self.tfidf_candidates = tfidf_candidates
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.num_negatives = num_negatives
        self.num_classes = len(corpus_texts)

    def __len__(self):
        return len(self.descriptions)

    def __getitem__(self, idx):
        query = self.descriptions[idx]
        pos_idx = self.naics_indices[idx]

        hard_pool = [int(c) for c in self.tfidf_candidates[idx] if int(c) != pos_idx]
        if len(hard_pool) >= self.num_negatives:
            neg_indices = random.sample(hard_pool[:50], self.num_negatives)
        else:
            neg_indices = list(hard_pool)
            while len(neg_indices) < self.num_negatives:
                r = random.randint(0, self.num_classes - 1)
                if r != pos_idx and r not in neg_indices:
                    neg_indices.append(r)

        candidate_indices = [pos_idx] + neg_indices

        input_ids_list = []
        attention_mask_list = []

        for c_idx in candidate_indices:
            enc = self.tokenizer(
                query, self.corpus_texts[c_idx],
                max_length=self.max_length, truncation=True,
                padding='max_length', return_tensors='pt'
            )
            input_ids_list.append(enc['input_ids'].squeeze(0))
            attention_mask_list.append(enc['attention_mask'].squeeze(0))

        return {
            'input_ids': torch.stack(input_ids_list),
            'attention_mask': torch.stack(attention_mask_list),
            'naics_idx': pos_idx,
        }

print("CrossEncoderDataset defined")

CrossEncoderDataset defined


In [ ]:
label_counts = df['naics_idx'].value_counts()
valid_labels = label_counts[label_counts >= 2].index
mask = df['naics_idx'].isin(valid_labels)
df_filtered = df[mask].reset_index(drop=True)
tfidf_filtered = tfidf_top100[mask.values]

train_idx, val_idx = train_test_split(
    np.arange(len(df_filtered)), test_size=VAL_RATIO, random_state=SEED, stratify=df_filtered['naics_idx']
)

train_dataset = CrossEncoderDataset(
    df_filtered['clean_description'].iloc[train_idx].tolist(),
    df_filtered['naics_idx'].iloc[train_idx].tolist(),
    corpus_texts,
    tfidf_filtered[train_idx],
    tokenizer, MAX_LENGTH, NUM_NEGATIVES
)

val_descriptions = df_filtered['clean_description'].iloc[val_idx].tolist()
val_naics_indices = df_filtered['naics_idx'].iloc[val_idx].tolist()
val_tfidf = tfidf_filtered[val_idx]

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

print(f"After filtering: {len(df_filtered)} samples, {df_filtered['naics_idx'].nunique()} codes")
print(f"Train: {len(train_dataset)} samples, {len(train_loader)} batches")
print(f"Val:   {len(val_descriptions)} samples")

val_tfidf_r50 = np.mean([val_naics_indices[i] in val_tfidf[i, :EVAL_TOP_K] for i in range(len(val_naics_indices))])
print(f"Val TF-IDF Recall@{EVAL_TOP_K}: {val_tfidf_r50:.4f} (ceiling for reranking eval)")

After filtering: 20533 samples, 1113 codes
Train: 18479 samples, 2310 batches
Val:   2054 samples
Val TF-IDF Recall@50: 0.5594 (ceiling for reranking eval)


In [ ]:
@torch.no_grad()
def evaluate_rerank(model, tokenizer, val_descriptions, val_labels, corpus_texts,
                    val_tfidf, max_length, eval_k=50, score_batch=64):
    model.eval()
    n = len(val_descriptions)
    top1, top5, top10 = 0, 0, 0
    recall_count = 0

    for i in range(n):
        query = val_descriptions[i]
        true_label = val_labels[i]
        candidates = val_tfidf[i, :eval_k].tolist()

        if true_label in candidates:
            recall_count += 1

        scores = []
        for j in range(0, len(candidates), score_batch):
            batch_cands = candidates[j:j+score_batch]
            batch_docs = [corpus_texts[int(c)] for c in batch_cands]
            enc = tokenizer(
                [query] * len(batch_docs), batch_docs,
                max_length=max_length, truncation=True, padding=True,
                return_tensors='pt'
            )
            enc = {k: v.to(device) for k, v in enc.items() if k in ['input_ids', 'attention_mask']}
            logits = model(**enc).logits.squeeze(-1)
            scores.extend(logits.cpu().tolist())

        ranked = sorted(zip(candidates, scores), key=lambda x: -x[1])
        ranked_indices = [int(r[0]) for r in ranked]

        if ranked_indices[0] == true_label: top1 += 1
        if true_label in ranked_indices[:5]: top5 += 1
        if true_label in ranked_indices[:10]: top10 += 1

    return {
        'tfidf_recall': recall_count / n,
        'top1': top1 / n,
        'top5': top5 / n,
        'top10': top10 / n,
    }


@torch.no_grad()
def evaluate_full_rerank(model, tokenizer, val_descriptions, val_labels, corpus_texts,
                         max_length, score_batch=128):
    model.eval()
    n = len(val_descriptions)
    num_classes = len(corpus_texts)
    top1, top5, top10 = 0, 0, 0

    for i in range(n):
        query = val_descriptions[i]
        true_label = val_labels[i]
        all_scores = []

        for j in range(0, num_classes, score_batch):
            batch_docs = corpus_texts[j:j+score_batch]
            enc = tokenizer(
                [query] * len(batch_docs), batch_docs,
                max_length=max_length, truncation=True, padding=True,
                return_tensors='pt'
            )
            enc = {k: v.to(device) for k, v in enc.items() if k in ['input_ids', 'attention_mask']}
            logits = model(**enc).logits.squeeze(-1)
            all_scores.append(logits.cpu())

        scores = torch.cat(all_scores, dim=0)
        topk = scores.topk(10).indices.tolist()

        if topk[0] == true_label: top1 += 1
        if true_label in topk[:5]: top5 += 1
        if true_label in topk[:10]: top10 += 1

        if (i + 1) % 200 == 0:
            print(f"  {i+1}/{n} | Top-1: {top1/(i+1):.4f}")

    return {'top1': top1 / n, 'top5': top5 / n, 'top10': top10 / n}

print("Evaluation functions defined")
print(f"  Per-epoch: rerank TF-IDF top-{EVAL_TOP_K} candidates")
print(f"  Final: rerank all {len(corpus_texts)} candidates")

Evaluation functions defined
  Per-epoch: rerank TF-IDF top-50 candidates
  Final: rerank all 1115 candidates


In [ ]:
from transformers import get_cosine_schedule_with_warmup

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

os.makedirs("results", exist_ok=True)
best_top1 = 0.0
best_epoch = 0
patience_counter = 0
epoch_log = []

print(f"Total steps: {total_steps}")
print(f"Warmup: {warmup_steps} steps")
print(f"\n{'='*80}")
print(f"{'Ep':>3} {'TrLoss':>8} {'TfR@50':>7} {'Top-1':>7} {'Top-5':>7} {'Top-10':>7} {'LR':>10}")
print(f"{'='*80}")

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    num_batches = 0

    for batch in train_loader:
        bsz = batch['input_ids'].size(0)
        num_cands = batch['input_ids'].size(1)

        input_ids = batch['input_ids'].view(-1, MAX_LENGTH).to(device)
        attention_mask = batch['attention_mask'].view(-1, MAX_LENGTH).to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits.squeeze(-1)
        scores = logits.view(bsz, num_cands)
        labels = torch.zeros(bsz, dtype=torch.long, device=device)
        loss = F.cross_entropy(scores, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        num_batches += 1

    avg_loss = total_loss / num_batches
    lr = scheduler.get_last_lr()[0]

    metrics = evaluate_rerank(
        model, tokenizer, val_descriptions, val_naics_indices,
        corpus_texts, val_tfidf, MAX_LENGTH, eval_k=EVAL_TOP_K
    )

    epoch_log.append({
        'epoch': epoch, 'train_loss': avg_loss, 'lr': lr,
        'tfidf_recall': metrics['tfidf_recall'],
        'top1': metrics['top1'], 'top5': metrics['top5'], 'top10': metrics['top10'],
    })

    print(f"{epoch:>3} {avg_loss:>8.4f} {metrics['tfidf_recall']:>7.4f} {metrics['top1']:>7.4f} "
          f"{metrics['top5']:>7.4f} {metrics['top10']:>7.4f} {lr:>10.2e}")

    if metrics['top1'] > best_top1:
        best_top1 = metrics['top1']
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), "results/best_model.pt")
    else:
        patience_counter += 1

    pd.DataFrame(epoch_log).to_csv("results/cross_encoder_epoch_log.csv", index=False)

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        break

print(f"\n{'='*80}")
print(f"Best Top-1 (TF-IDF@{EVAL_TOP_K} rerank): {best_top1:.4f} at epoch {best_epoch}")
model.load_state_dict(torch.load("results/best_model.pt", weights_only=True))
print("Loaded best model.")

Total steps: 34650
Warmup: 3465 steps

 Ep   TrLoss  TfR@50   Top-1   Top-5  Top-10         LR
  1   1.7933  0.5594  0.1748  0.3890  0.4722   1.33e-05
  2   1.2743  0.5594  0.2020  0.4065  0.4854   1.99e-05
  3   1.0052  0.5594  0.2030  0.4148  0.4893   1.94e-05
  4   0.7814  0.5594  0.2152  0.4231  0.5024   1.84e-05
  5   0.6250  0.5594  0.2113  0.4323  0.5044   1.69e-05
  6   0.5264  0.5594  0.2157  0.4216  0.4966   1.50e-05
  7   0.4555  0.5594  0.2137  0.4260  0.5000   1.29e-05
  8   0.3939  0.5594  0.1977  0.4289  0.5005   1.06e-05
  9   0.3546  0.5594  0.2084  0.4323  0.4995   8.26e-06
 10   0.3136  0.5594  0.2089  0.4357  0.5019   6.04e-06
 11   0.2819  0.5594  0.2040  0.4299  0.4961   4.03e-06

Early stopping at epoch 11

Best Top-1 (TF-IDF@50 rerank): 0.2157 at epoch 6
Loaded best model.


In [ ]:
print("=== Final Evaluation: Reranking ALL 1,115 NAICS codes ===")
print("(This takes ~15 minutes)\n")

full_metrics = evaluate_full_rerank(
    model, tokenizer, val_descriptions, val_naics_indices,
    corpus_texts, MAX_LENGTH, score_batch=128
)

tfidf_metrics = evaluate_rerank(
    model, tokenizer, val_descriptions, val_naics_indices,
    corpus_texts, val_tfidf, MAX_LENGTH, eval_k=EVAL_TOP_K
)

print(f"\n=== Results ===")
print(f"\n  Full Rerank (all {len(corpus_texts)} candidates):")
print(f"    Top-1:  {full_metrics['top1']:.4f}")
print(f"    Top-5:  {full_metrics['top5']:.4f}")
print(f"    Top-10: {full_metrics['top10']:.4f}")
print(f"\n  TF-IDF@{EVAL_TOP_K} Rerank:")
print(f"    Recall: {tfidf_metrics['tfidf_recall']:.4f}")
print(f"    Top-1:  {tfidf_metrics['top1']:.4f}")
print(f"    Top-5:  {tfidf_metrics['top5']:.4f}")
print(f"    Top-10: {tfidf_metrics['top10']:.4f}")
print(f"\n  Best epoch: {best_epoch}")

results = {
    "method": "DeBERTa-v3-small Cross-Encoder + TF-IDF Candidate Retrieval",
    "config": {
        "model": MODEL_NAME, "max_length": MAX_LENGTH, "batch_size": BATCH_SIZE,
        "num_negatives": NUM_NEGATIVES, "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY, "num_epochs_trained": best_epoch,
        "eval_top_k": EVAL_TOP_K,
    },
    "full_rerank_results": full_metrics,
    "tfidf_rerank_results": tfidf_metrics,
}

with open("results/cross_encoder_results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)

import zipfile, glob
with zipfile.ZipFile("results.zip", "w") as zf:
    for f in glob.glob("results/*.json") + glob.glob("results/*.csv") + glob.glob("results/*.pt"):
        zf.write(f)
        print(f"  Added: {f}")

from google.colab import files
files.download("results.zip")
print("\nDownloaded results.zip")

=== Final Evaluation: Reranking ALL 1,115 NAICS codes ===
(This takes ~15 minutes)

  200/2054 | Top-1: 0.0750
  400/2054 | Top-1: 0.0625
  600/2054 | Top-1: 0.0617
  800/2054 | Top-1: 0.0563
  1000/2054 | Top-1: 0.0540
  1200/2054 | Top-1: 0.0525
  1400/2054 | Top-1: 0.0543
  1600/2054 | Top-1: 0.0519
  1800/2054 | Top-1: 0.0556
  2000/2054 | Top-1: 0.0550

=== Results ===

  Full Rerank (all 1115 candidates):
    Top-1:  0.0545
    Top-5:  0.1207
    Top-10: 0.1680

  TF-IDF@50 Rerank:
    Recall: 0.5594
    Top-1:  0.2157
    Top-5:  0.4216
    Top-10: 0.4966

  Best epoch: 6
  Added: results/cross_encoder_results.json
  Added: results/cross_encoder_epoch_log.csv
  Added: results/best_model.pt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Downloaded results.zip
